# Phase 4.2：从 KnowledgeBase 到 evidence-first 服务

## 目标

先把服务内部逻辑拆成“导入 → 建索引 → 搜索 → 组织证据”，再通过 FastAPI 暴露。理解这一层，才能知道 API 路由不是魔法，而是对已有项目能力的编排。

**本课交付：** 一次临时知识库导入记录和 evidence-only 答案验证。

## Evidence Quest 任务卡：Phase 4.2：搭建证据工作台

**你的身份：** Mini RAG 产品工程师  
**案件背景：** 现在把前面所有证据接到一张工作台：用户提出问题，系统检索原文，并把能点击回去的证据交还给用户。

### 本关专业 Goal

完成一个无外部 API Key 也能工作的 evidence-first Mini RAG 服务。

### 你要交付的作品

**可运行的 Evidence Desk API**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：证据工作台建造师  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. 为什么索引不能每次请求重建？

文档解析和 BM25 建索引是相对昂贵的初始化工作。服务启动时 ingest 一次，之后多个 Query 复用同一个 `KnowledgeBase`；如果每次请求都重建，延迟会变高，也更难保证同一批请求使用同一版本索引。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase4.2'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase4.2
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 导入临时目录工具，创建一个隔离的练习知识库。
from tempfile import TemporaryDirectory

# 导入 KnowledgeBase，观察服务内部的核心对象。
from phase4_mini_rag_system.knowledge_base import KnowledgeBase

# 创建临时目录，退出代码块后会自动清理。
with TemporaryDirectory() as temporary_directory:
    # 把临时目录字符串转换成 Path 对象。
    temporary_path = Path(temporary_directory)

    # 指定一份临时 Markdown 文件。
    guide_path = temporary_path / "guide.md"

    # 写入一条可被检索的事实。
    guide_path.write_text("# Guide\n\noverlap 保留跨边界上下文。", encoding="utf-8")

    # 创建一个空 KnowledgeBase。
    knowledge_base = KnowledgeBase()

    # 导入文档并构建 BM25 索引。
    chunk_count = knowledge_base.ingest(temporary_path, chunk_size=128, overlap=32)

    # 打印导入数量和索引版本。
    print("ingested chunks:", chunk_count)
    print("index version:", knowledge_base.index_version)

    # 确认索引已经准备好。
    assert chunk_count > 0
    assert knowledge_base.retriever is not None

ingested chunks: 1
index version: chunks-1-size-128-overlap-32


## 2. 搜索和证据回答是两个步骤

`search()` 负责找到并排序证据；`evidence_answer()` 负责在没有 LLM 时把证据明确展示给用户。分开这两步有两个好处：检索可以独立评估，生成失败时仍然保留证据。

In [4]:
# 再创建一个临时目录，保证本单元格可以独立执行。
with TemporaryDirectory() as temporary_directory:
    # 把临时目录转换为 Path。
    temporary_path = Path(temporary_directory)

    # 创建知识库输入文件。
    (temporary_path / "guide.md").write_text("Chunk overlap 可以保留跨边界上下文。", encoding="utf-8")

    # 创建并导入知识库。
    knowledge_base = KnowledgeBase()
    knowledge_base.ingest(temporary_path)

    # 搜索用户问题并取得结构化结果。
    results = knowledge_base.search("overlap 上下文", top_k=3)

    # 将搜索结果组织成 evidence-only 答案。
    answer = knowledge_base.evidence_answer("overlap 上下文", results)

    # 输出答案和第一条引用信息。
    print(answer)
    print(results[0])

    # 有证据的问题必须返回结果和 Chunk ID。
    assert results
    assert results[0]["chunk_id"]

    # evidence-only 答案必须明确它没有调用生成模型。
    assert "evidence-only" in answer

当前为 evidence-only 模式。请依据以下可追溯证据作答：
[90cc62c4aedd99cc] Chunk overlap 可以保留跨边界上下文。
{'chunk_id': '90cc62c4aedd99cc', 'text': 'Chunk overlap 可以保留跨边界上下文。', 'source': 'C:\\Users\\FFY\\AppData\\Local\\Temp\\tmpfrtd44v7\\guide.md', 'page': None, 'score': 1.150728, 'metadata': {'source': 'C:\\Users\\FFY\\AppData\\Local\\Temp\\tmpfrtd44v7\\guide.md', 'page': None, 'chunk_index': 0, 'metadata': {'format': 'markdown', 'headings': []}}}


## 3. 没有证据时不能编造

对不存在的词进行查询，结果应该为空，答案应该明确说没有足够证据。这个行为是 evidence-first 产品的安全底线。

In [5]:
# 创建一个包含已知事实的临时知识库。
with TemporaryDirectory() as temporary_directory:
    # 把临时目录转为 Path。
    temporary_path = Path(temporary_directory)

    # 写入一条不包含未知词的文档。
    (temporary_path / "guide.md").write_text("系统支持本地检索。", encoding="utf-8")

    # 导入文档并建立索引。
    knowledge_base = KnowledgeBase()
    knowledge_base.ingest(temporary_path)

    # 查询知识库中不存在的词。
    empty_results = knowledge_base.search("完全不存在的词", top_k=3)

    # 让服务根据空证据生成 fallback 文本。
    empty_answer = knowledge_base.evidence_answer("完全不存在的词", empty_results)

    # 输出结果，观察系统如何表达不知道。
    print("results:", empty_results)
    print("answer:", empty_answer)

    # 没有证据时结果必须为空。
    assert empty_results == []

    # 答案必须明确表示无法回答。
    assert "无法回答" in empty_answer

results: []
answer: 当前知识库没有找到足够证据，无法回答该问题。


## 本课验收

- [ ] 能画出 ingest、search、answer 的内部顺序。
- [ ] 能解释为什么索引应该复用。
- [ ] 有证据时返回引用，无证据时明确不知道。
- [ ] 能说明 evidence-only fallback 如何降低幻觉风险。

## Boss Challenge：提出一个没有证据的问题，确认系统返回不知道，而不是编造答案。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [6]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [7]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase4_service_record.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: 还没有生成，请回到交付单元格
